# Telco Customer Churn — EDA y Entrenamiento

Exploración del dataset, limpieza, análisis y entrenamiento del modelo de predicción de churn.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report,
    ConfusionMatrixDisplay, confusion_matrix,
)

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

ROOT = Path('..').resolve()
DATA_PATH   = ROOT / 'data'   / 'WA_Fn-UseC_-Telco-Customer-Churn.csv'
MODELS_PATH = ROOT / 'models'

## 2. Carga y primera inspección

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f'Filas: {df.shape[0]}   Columnas: {df.shape[1]}')
df.head()

In [ ]:
df.dtypes

In [ ]:
df.isnull().sum()

## 3. Limpieza

`TotalCharges` viene como `object` en vez de `float`. Los clientes con `tenure = 0` tienen un espacio en blanco en esa columna (nunca pagaron nada), que hay que convertir a `0`.

In [ ]:
# Clientes con TotalCharges en blanco
blank_mask = df['TotalCharges'].str.strip() == ''
print(f'Registros con TotalCharges en blanco: {blank_mask.sum()}')
df[blank_mask][['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges']].head()

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].str.strip(), errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0.0)
df['TotalCharges'].dtype

In [ ]:
df = df.drop(columns=['customerID'])
df['Churn'] = (df['Churn'] == 'Yes').astype(int)
print('Churn únicos:', df['Churn'].unique())

## 4. Análisis exploratorio

In [ ]:
churn_counts = df['Churn'].value_counts()
churn_rate   = df['Churn'].mean()
print(f'No Churn: {churn_counts[0]}  ({1 - churn_rate:.1%})')
print(f'Churn:    {churn_counts[1]}  ({churn_rate:.1%})')

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(['No Churn', 'Churn'], churn_counts.values, color=['#4C72B0', '#DD8452'])
ax.set_title('Distribución de Churn')
ax.set_ylabel('Clientes')
for i, v in enumerate(churn_counts.values):
    ax.text(i, v + 30, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col in zip(axes, ['tenure', 'MonthlyCharges', 'TotalCharges']):
    df.groupby('Churn')[col].plot(kind='kde', ax=ax, legend=True)
    ax.set_title(f'{col} por Churn')
    ax.legend(['No Churn', 'Churn'])

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
cat_preview = ['Contract', 'InternetService', 'PaymentMethod',
               'TechSupport', 'OnlineSecurity', 'PaperlessBilling']

for ax, col in zip(axes.flat, cat_preview):
    ct = df.groupby(col)['Churn'].mean().sort_values(ascending=False)
    ct.plot(kind='bar', ax=ax, color='#4C72B0')
    ax.set_title(f'Churn rate por {col}')
    ax.set_ylabel('Churn rate')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')

plt.tight_layout()
plt.show()

## 5. Preparación de features

In [ ]:
X = df.drop(columns=['Churn'])
y = df['Churn']

cat_cols = X.select_dtypes(include='object').columns.tolist()
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

print(f'Numéricas  ({len(num_cols)}): {num_cols}')
print(f'Categóricas ({len(cat_cols)}): {cat_cols}')

## 6. Train / Test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {len(X_train)}  |  Test: {len(X_test)}')
print(f'Churn rate train: {y_train.mean():.2%}  |  test: {y_test.mean():.2%}')

## 7. Modelo

Pipeline: `StandardScaler` + `OneHotEncoder` → `RandomForestClassifier`.

`class_weight='balanced'` para compensar el desbalance de clases (~27% churn).

In [ ]:
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
])

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=200,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1,
    )),
])

pipeline.fit(X_train, y_train)
print('Entrenamiento completado')

## 8. Evaluación

In [ ]:
y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

metrics = {
    'accuracy':  accuracy_score(y_test, y_pred),
    'precision': precision_score(y_test, y_pred),
    'recall':    recall_score(y_test, y_pred),
    'f1':        f1_score(y_test, y_pred),
    'roc_auc':   roc_auc_score(y_test, y_prob),
}

for k, v in metrics.items():
    print(f'{k:<12} {v:.4f}')

In [ ]:
print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred),
    display_labels=['No Churn', 'Churn']
).plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Matriz de Confusión')
plt.tight_layout()
plt.show()

## 9. Importancia de features

In [ ]:
ohe_cols = pipeline.named_steps['preprocessor']\
    .named_transformers_['cat']\
    .get_feature_names_out(cat_cols).tolist()
all_feature_names = num_cols + ohe_cols

importances = pipeline.named_steps['classifier'].feature_importances_
feat_imp = pd.Series(importances, index=all_feature_names).nlargest(15)

fig, ax = plt.subplots(figsize=(8, 5))
feat_imp.sort_values().plot(kind='barh', ax=ax, color='#4C72B0')
ax.set_title('Top 15 features más importantes')
ax.set_xlabel('Importancia')
plt.tight_layout()
plt.show()

## 10. Guardar modelo

In [ ]:
MODELS_PATH.mkdir(exist_ok=True)

joblib.dump(pipeline, MODELS_PATH / 'churn_pipeline.pkl')

metadata = {
    'model_type': 'RandomForestClassifier',
    'features': X.columns.tolist(),
    'categorical_features': cat_cols,
    'numerical_features': num_cols,
    'metrics': {k: round(v, 4) for k, v in metrics.items()},
    'target': 'Churn',
    'classes': ['No Churn', 'Churn'],
    'train_samples': len(X_train),
    'test_samples': len(X_test),
}

with open(MODELS_PATH / 'model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(f'Modelo guardado: {MODELS_PATH / "churn_pipeline.pkl"}')
print(f'Metadata guardada: {MODELS_PATH / "model_metadata.json"}')